# Telco Customer Churn Prediction
Xinyu Jia  |  C00304524


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve


In [ ]:
from google.colab import files
files.upload()

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()


In [ ]:
print(df.shape)
df.info()


In [ ]:
# drop ID and fix TotalCharges
df = df.drop('customerID', axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

print('After cleaning:', df.shape)
print(df['Churn'].value_counts())


In [ ]:
sns.countplot(x='Churn', data=df)
plt.title('Customer Churn Distribution')
plt.show()


In [ ]:
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title('Churn by Contract Type')
plt.show()


In [ ]:
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges by Churn')
plt.show()


In [ ]:
sns.countplot(x='PaymentMethod', hue='Churn', data=df)
plt.title('Churn by Payment Method')
plt.xticks(rotation=20)
plt.show()


In [ ]:
# target: Yes -> 1, No -> 0
y = (df['Churn'] == 'Yes').astype(int)
X = pd.get_dummies(df.drop('Churn', axis=1), drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)


In [ ]:
# correlation among numeric features
num_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
plt.figure(figsize=(7, 5))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()

# correlation of each encoded feature with Churn
corr_churn = X.corrwith(y).sort_values()
plt.figure(figsize=(8, 8))
corr_churn.plot(kind='barh', color='steelblue')
plt.title('Feature Correlation with Churn')
plt.axvline(0, color='k', linewidth=0.8)
plt.show()


In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]


In [ ]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))


In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label='Random Forest')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot(kind='barh', figsize=(8, 7))
plt.title('Feature Importance')
plt.show()
